# appflame: finding and testing growth opportunities in Hily and Taimi

One notebook, in the order of the job: **look at the market → listen to users → find opportunities → turn them into hypotheses →
design and read tests → model churn**. Two kinds of data, always labelled:

- **PUBLIC DATA**: App Store (Apple APIs and pages), written reviews (aggregates only), Google Trends, ECB rates, the Columbia speed-dating experiment.
- **SIMULATED**: "App H", a fictional dating app (200,000 sign-ups, 6 months, seed 42). Never appflame, Hily or Taimi data.

Run order: `src/download_data.py` → `src/simulate.py` → `src/analysis_public.py` → `src/analysis_product.py` → this notebook.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
import ab_stats as ab
from common import OUT, PUB, sim_con
from run_sql import run
pd.set_option("display.precision", 3); pd.set_option("display.width", 140)
P = json.loads((OUT / "public_results.json").read_text()); S = json.loads((OUT / "product_results.json").read_text())
con = sim_con()

## 1. Hily and Taimi in public data (PUBLIC DATA)
Published facts are quoted *as stated by the company*, with the page and the date.

In [2]:
pd.read_csv(PUB / "company_facts.csv")[["entity", "fact", "value", "seen_on"]]

,entity,fact,value,seen_on
0,appflame,Description,Ukrainian product IT company that creates Taim...,2026-11-15
1,appflame,Company age,8 years,2026-11-15
2,appflame,Team size,550+ specialists worldwide,2026-11-15
3,appflame,Products,"Hily (dating), Taimi (inclusive LGBTQ+ dating)...",2026-11-15
4,appflame,Stated goals,Reach the top 5 product companies in the world...,2026-11-15
5,appflame,Market position,Hily and Taimi in the top 5 dating apps in the...,2026-11-15
6,Hily,Name origin,"Hily comes from 'Hey, I Like You!'",2026-11-15
7,Hily,Likes sent,11.9B+,2026-11-15
8,Hily,Downloads,44M+ globally,2026-11-15
9,Hily,Mutual matches,58M+ annually,2026-11-15


In [3]:
r = pd.read_csv(OUT / "pub_ratings_us.csv")[["app", "avg_rating", "rating_count", "current_version_date"]]
r.sort_values("rating_count", ascending=False)

,app,avg_rating,rating_count,current_version_date
0,Plenty of Fish,4.286,1809706,2026-11-03
1,Bumble,4.317,1772309,2026-11-09
2,Tinder,4.200,1759872,2026-11-11
3,Hinge,4.405,1141688,2026-11-13
4,Hily,4.411,391836,2026-11-12
5,Grindr,4.450,390084,2026-11-12
6,OkCupid,4.319,289884,2026-10-21
7,Match,3.865,245573,2026-11-03
8,Taimi,4.419,217902,2026-11-10
9,Badoo,4.577,150275,2026-11-09


Same item, six storefronts, converted with ECB rates. GB, DE and FR prices include VAT; US, CA and AU store prices exclude sales tax.

In [4]:
pd.read_csv(OUT / "pub_price_by_country.csv")

,app,item,country,price_local,currency,price_usd,vat_included
0,Hily,1-week Hily Premium,AU,24.99,AUD,16.96,False
1,Hily,1-week Hily Premium,CA,19.99,CAD,14.02,False
2,Hily,1-week Hily Premium,DE,17.49,EUR,19.33,True
3,Hily,1-week Hily Premium,FR,17.99,EUR,19.88,True
4,Hily,1-week Hily Premium,GB,14.99,GBP,19.83,True
5,Hily,1-week Hily Premium,US,14.99,USD,14.99,False
6,Taimi,Taimi Gold Subscription,AU,24.99,AUD,16.96,False
7,Taimi,Taimi Gold Subscription,CA,22.99,CAD,16.12,False
8,Taimi,Taimi Gold Subscription,DE,19.99,EUR,22.09,True
9,Taimi,Taimi Gold Subscription,FR,19.99,EUR,22.09,True


In [5]:
print(json.dumps(P["trends"], indent=1)); print("release rhythm (versions seen in reviews per 30 days):", P["reviews"]["cadence"])

{
 "hily_us_2026": 4.6521739130434785,
 "hily_us_2023": 3.207547169811321,
 "taimi_us_2026": 3.717391304347826,
 "taimi_us_2023": 2.018867924528302,
 "grindr_us_2026": 43.06521739130435,
 "her_us_2026": 21.652173913043477,
 "note": "Relative search interest (0-100) within one request; 'Taimi' and 'Hily' are also first names."
}
release rhythm (versions seen in reviews per 30 days): {'Bumble': 4.0, 'Hily': 5.3, 'Hinge': 2.0, 'Taimi': 2.7, 'Tinder': 3.3}


## 2. What users say (PUBLIC DATA)
Transparent keyword rules (`src/analysis_public.py`, `THEMES`), multi-label. Checked on 45 reviews labelled by hand
(`data/public/review_validation_labels.csv`, ids only). Written reviews are a negative, self-selected sample: read **themes**, not satisfaction.

In [6]:
pd.read_csv(OUT / "pub_review_validation.csv")

,theme,manual,predicted,tp,precision,recall
0,Paywall & price,15,21,15,0.714,1.000
1,Billing & refunds,4,4,4,1.000,1.000
2,Fake profiles & scams,9,10,8,0.800,0.889
3,"Few matches, distance, reach",7,6,6,1.000,0.857
4,Bans & blocked accounts,4,4,4,1.000,1.000
5,Bugs & technical,2,2,2,1.000,1.000
6,Ads,3,5,3,0.600,1.000
7,Support,2,2,1,0.500,0.500
8,Positive experience,14,12,12,1.000,0.857


In [7]:
th = pd.read_csv(OUT / "pub_review_themes.csv").pivot(index="theme", columns="app", values="share")
th["peers"] = th[["Tinder", "Bumble", "Hinge"]].mean(axis=1)
th[["Hily", "Taimi", "peers"]].sort_values("Hily", ascending=False).style.format("{:.0%}")

app,Hily,Taimi,peers
theme,,,
Paywall & price,43%,53%,32%
Positive experience,20%,6%,18%
Fake profiles & scams,19%,26%,15%
"Few matches, distance, reach",9%,12%,9%
Billing & refunds,7%,6%,5%
Bans & blocked accounts,5%,5%,11%
Ads,5%,3%,1%
Support,5%,6%,10%
Bugs & technical,4%,5%,9%


## 3. What makes a match mutual (PUBLIC DATA, speed-dating experiment)
One row per pair, target = both said yes. Three feature sets: profile only → + behaviour history (leave-one-out yes-rates) → + ratings
after meeting. Cross-validation holds out whole sessions (`GroupKFold`), so no person is in both train and test.
**Not used:** race, religion, origin, income, gender.

In [8]:
sd = P["speed_dating"]
print(f"yes rate {sd['yes_rate']:.1%} | match rate {sd['match_rate']:.1%} | if independent {sd['independent_match']:.1%} | corr(dec, dec_o) {sd['corr_dec']:+.3f}")
print(f"top 10% most-liked people receive {sd['top10_share_yes']:.0%} of all yeses")
pd.read_csv(OUT / "pub_sd_models.csv")

yes rate 42.0% | match rate 16.5% | if independent 17.6% | corr(dec, dec_o) -0.047
top 10% most-liked people receive 23% of all yeses


,model,auc_lr,auc_gb,n_features
0,profile,0.598,0.526,7
1,profile_history,0.733,0.673,11
2,after_meeting,0.843,0.846,23


In [9]:
pd.read_csv(OUT / "pub_sd_coefs.csv").head(8)  # standardised logistic coefficients, full model

,feature,coef_std
0,picky_loo,0.596
1,attr_o,0.591
2,attr,0.504
3,fun,0.477
4,shar,0.416
5,picky_loo_p,0.386
6,shar_o,0.268
7,fun_o,0.267


In [10]:
pd.read_csv(OUT / "pub_sd_by_shared.csv")

,lower_shared_interest_rating,match_rate,pairs
0,0-3,0.054,953
1,4-5,0.133,1088
2,6-7,0.326,711
3,8-10,0.554,166


**Product translation.** (1) Rank pairs on *both* sides' probability of saying yes, using like-back history: behaviour adds far more than
profile fields. (2) Attention concentrates naturally: new profiles need guaranteed exposure. (3) *Perceived* shared interests drive
matches much more than measured overlap: prompts and ice-breakers that surface common ground (tested in E6).

## 4. Where the growth is (SIMULATED)
Each opportunity comes from one SQL query in `sql/` and ends as a testable hypothesis.

In [11]:
run("01_onboarding_funnel.sql", con)

,platform,step,users,share_of_signups,pass_from_previous
0,android,1_basic_info,85179,1.000,NaN
1,android,2_birthday,78435,0.921,0.921
2,android,3_interests,76086,0.893,0.970
3,android,4_photo_upload,71497,0.839,0.940
4,android,5_bio,47202,0.554,0.660
5,android,6_location,44881,0.527,0.951
6,ios,1_basic_info,114821,1.000,NaN
7,ios,2_birthday,106894,0.931,0.931
8,ios,3_interests,103657,0.903,0.970
9,ios,4_photo_upload,97497,0.849,0.941


In [12]:
run("06_like_balance.sql", con)

,likes_bucket,users,share_of_users,return_d1,return_week1,share_side_a,top10pct_share_of_likes
0,0,11769,0.106,0.211,0.633,0.978,0.361
1,1-2,34093,0.306,0.404,0.856,0.932,0.361
2,3-5,28909,0.260,0.409,0.859,0.713,0.361
3,6-10,20159,0.181,0.452,0.885,0.238,0.361
4,11+,16304,0.147,0.443,0.887,0.020,0.361


In [13]:
run("07_push_hour.sql", con).iloc[[0, 4, 11, 12, 13, 14]]

,local_hour,pushes,open_rate,return_rate,open_rate_moe95
0,8,86686,0.043,0.023,0.001
4,12,85182,0.060,0.034,0.002
11,19,82755,0.085,0.047,0.002
12,20,83067,0.086,0.048,0.002
13,21,82851,0.087,0.047,0.002
14,22,85492,0.054,0.030,0.002


In [14]:
run("08_channel_quality.sql", con)

,channel,signups,cost_per_signup,active_d28_32,payer_share,revenue_per_signup,roas_to_date
0,apple_search_ads,14149,4.77,0.358,0.110,6.78,1.42
1,organic,31388,0.00,0.326,0.088,5.72,NaN
2,meta,16171,4.13,0.290,0.067,4.28,1.04
3,influencer,7118,3.49,0.269,0.056,3.51,1.01
4,google_uac,10227,3.17,0.249,0.055,3.39,1.07
5,tiktok,10974,2.39,0.209,0.033,1.89,0.79


In [15]:
for k, o in S["opportunities"].items():
    print(f"[{k}] {o['hypothesis']}\n    primary: {o['metric']} | guardrails: {o['guardrails']}"
          + (f" | n per arm: {o['n_per_arm']:,}" if "n_per_arm" in o else "") + (f" | design: {o['design']}" if "design" in o else "") + "\n")

[photo] If the Android photo step accepts any gallery format and lets users add a photo later, then onboarding completion on Android will rise toward iOS level, because 34% of Android users leave exactly at that step while iOS users pass it at 89%.
    primary: onboarding completion (Android) | guardrails: profiles with >= 1 photo after 7 days; report rate; likes received per profile | n per arm: 5,543

[likes] If every new profile is guaranteed a minimum exposure on day 0 (a fair-exposure slot in the card stack), then day-1 return will rise, because new users who receive no like on day 0 come back half as often (21% vs 42%).
    primary: day-1 return of new users | guardrails: likes received by existing users; match rate; reports | n per arm: 16,124 | design: two-sided: randomise by metro (cluster), see test E3

[push] If re-engagement pushes are sent between 19:00 and 21:00 local time instead of a fixed spread over the day, then same-day return will rise from 3.2% to about 4.7%, beca

## 5. A/B tests, end to end (SIMULATED)
### 5.1 Size before launch (E1 paywall price)
n per arm = 2 (z₀.₉₇₅ + z₀.₈)² σ² / δ², with σ and the mean taken from a pre-period without any test.

In [16]:
base = run("05_revenue_per_user_payer.sql", con)
pre = S["E1"]["pre"]
for mde in (0.05, 0.08, 0.12):
    n = ab.sample_size_mean(pre["mean"], pre["sd"], mde)
    print(f"MDE {mde:.0%}: {n:,} users per arm = {2 * n / pre['signups_per_day']:.0f} days of traffic")

MDE 5%: 181,610 users per arm = 396 days of traffic
MDE 8%: 70,942 users per arm = 155 days of traffic
MDE 12%: 31,530 users per arm = 69 days of traffic


### 5.2 Read E1 on the pre-registered metric (30-day revenue), not on day 1

In [17]:
e1 = run("09_experiment_readout.sql", con)
t, c = e1[e1.variant == "treatment"], e1[e1.variant == "control"]
rows = {m: ab.mean_test(t[m], c[m]) for m in ("rev_d1", "rev_d7", "rev_d30")}
rows["paid_d30"] = ab.prop_test(t.paid_d30.sum(), len(t), c.paid_d30.sum(), len(c))
pd.DataFrame(rows).T[["rel", "rel_low", "rel_high", "p"]]

,rel,rel_low,rel_high,p
rev_d1,0.259,0.114,0.405,1.113e-04
rev_d7,0.240,0.121,0.360,1.566e-05
rev_d30,0.204,0.103,0.304,1.754e-05
paid_d30,-0.102,-0.169,-0.036,4.193e-03


In [18]:
pd.read_csv(OUT / "p_e1_weekly.csv")  # novelty: week 1 conversion is up, then down

,week,users,conv_rel,rev_d30_rel,rev_d30_low,rev_d30_high
0,1,7282,-0.068,0.272,-0.025,0.570
1,2,7543,-0.172,0.120,-0.145,0.385
2,3,7741,-0.030,0.235,-0.072,0.543
3,4,8190,-0.050,0.304,-0.015,0.622
4,5,8232,-0.093,0.224,-0.068,0.516
5,6,8450,-0.066,0.253,-0.044,0.550
6,7,8618,-0.061,0.257,-0.025,0.540
7,8,8581,-0.246,0.005,-0.226,0.237


### 5.3 Check the split first: sample-ratio mismatch (E2 onboarding)

In [19]:
srm = run("10_srm_check.sql", con)
srm["p"] = [ab.srm_test(a, b)["p"] for a, b in zip(srm.n_t, srm.n_c)]
srm[srm.experiment_id == "E2_onboarding"]

,experiment_id,platform,n_t,n_c,share_treatment,chi2,p
3,E2_onboarding,all,12154,12832,0.486,18.40,1.793e-05
4,E2_onboarding,android,4917,5548,0.470,38.05,6.906e-10
5,E2_onboarding,ios,7237,7284,0.498,0.15,6.965e-01


In [20]:
e2 = S["E2"]
pd.DataFrame({k: e2[k] for k in ("naive", "ios_only", "itt")}).T[["diff", "ci_low", "ci_high", "p"]]

,diff,ci_low,ci_high,p
naive,0.057,0.045,0.068,0.000e+00
ios_only,0.025,0.010,0.040,8.960e-04
itt,0.029,0.017,0.041,1.000e-06


Assignment is a deterministic hash of the user id, so the intended group of every sign-up can be rebuilt (intent-to-treat), including
Android users whose exposure was never logged. Raw read ≈ +5.7 pts; honest read ≈ +3 pts.

### 5.4 Two-sided market: interference (E3 smart picks)

In [21]:
m = run("11_interference_by_design.sql", con)
print(json.dumps({k: S["E3"][k] for k in ("cluster", "spillover_control_drop")}, indent=1))
print("user-split read:", {k: round(S["E3"]["naive"][k], 4) for k in ("rel", "rel_low", "rel_high", "p")})
m.groupby(["design", "variant"])[["users"]].sum()

{
 "cluster": {
  "rel": 0.034084,
  "rel_low": -0.006361,
  "rel_high": 0.074529,
  "p": 0.093609,
  "n_metros_t": 11,
  "n_metros_c": 9
 },
 "spillover_control_drop": -0.08041
}
user-split read: {'rel': 0.2351, 'rel_low': 0.1959, 'rel_high': 0.2743, 'p': 0.0}


users
design variant         
metro  control    17716
       treatment  22680
user   control    20536
       treatment  20287

### 5.5 Five tests at once (E4): multiple-testing correction

In [22]:
e4 = pd.read_csv(OUT / "p_e4.csv")
e4[["experiment", "rel", "p", "p_holm", "p_bh"]]

,experiment,rel,p,p_holm,p_bh
0,E4a_badge_color,0.006,0.540,1.000,0.808
1,E4b_profile_tips,-0.004,0.701,1.000,0.808
2,E4c_dark_paywall,0.022,0.029,0.143,0.143
3,E4d_swipe_hint,0.002,0.808,1.000,0.808
4,E4e_new_icons,-0.003,0.777,1.000,0.808


### 5.6 Peeking (E5): daily reads vs a fixed horizon vs an O'Brien-Fleming boundary

In [23]:
e5 = pd.read_csv(OUT / "p_e5_daily.csv")
print(e5.loc[e5.p < 0.05, ["day", "rel", "p"]])
print("final:", e5.iloc[-1][["day", "rel", "p"]].to_dict())
ab.peeking_fpr(n_per_day=600, days=28, p_base=0.57, n_sims=500)   # A/A simulations (500 here for speed; 2,000 in the report)

    day    rel      p
6     7  0.040  0.047
10   11  0.033  0.036
11   12  0.033  0.029
final: {'day': 28.0, 'rel': -0.0048810106461091, 'p': 0.6106349561464259}


{'naive_daily': np.float64(0.274),
 'single_look': np.float64(0.044),
 'obf_daily': np.float64(0.046),
 'n_sims': 500}

### 5.7 CUPED (E6 ice-breakers): same decision with fewer users

In [24]:
d6 = run("12_cuped_inputs.sql", con)
cu = ab.cuped(d6.y_messages, d6.x_pre_messages, d6.variant == "treatment")
print(f"theta = {cu['theta']:.3f}, rho = {cu['rho']:.3f}, variance reduction = {cu['var_reduction']:.1%}")
pd.DataFrame({"raw": cu["raw"], "CUPED": cu["adjusted"]}).T[["rel", "rel_low", "rel_high", "p"]]

theta = 0.493, rho = 0.544, variance reduction = 29.6%


,rel,rel_low,rel_high,p
raw,0.050,0.019,0.082,1.347e-03
CUPED,0.054,0.027,0.080,4.833e-05


## 6. Predictive model: who cancels at the first renewal (SIMULATED)
Features from the first 7 days of the subscription only (`sql/13`), 5-fold stratified CV.

In [25]:
print(json.dumps(S["churn"], indent=1)); pd.read_csv(OUT / "p_churn_by_matches.csv")

{
 "n": 9215,
 "base_rate": 0.367227,
 "auc_lr": 0.676437,
 "auc_gb": 0.658456,
 "brier_lr": 0.211798,
 "brier_gb": 0.216538,
 "top_decile_rate": 0.616052,
 "bottom_decile_rate": 0.137744,
 "top_coefs": [
  "matches_7d",
  "active_days_7d",
  "plan_weekly",
  "trial",
  "price_usd",
  "country_US"
 ],
 "churn_0_matches": 0.51514,
 "churn_6plus": 0.192683
}


,matches_7d_bucket,churn_rate,subs
0,0,0.515,2543
1,1-2,0.391,3156
2,3-5,0.279,1876
3,6+,0.193,1640


In [26]:
pd.read_csv(OUT / "p_churn_coefs.csv").reindex(pd.read_csv(OUT / "p_churn_coefs.csv").coef_std.abs().sort_values(ascending=False).index).head(8)

,feature,coef_std
0,matches_7d,-0.493
1,active_days_7d,-0.275
18,plan_weekly,0.260
17,trial,0.207
16,price_usd,0.092
2,country_US,-0.087
3,country_GB,-0.066
15,messages_7d,0.044


## 7. Last step: compare with the hidden truth
Opened only now, after every estimate above was written.

In [27]:
pd.read_csv(ROOT / "data" / "_hidden_truth.csv")

,item,metric,true_effect
0,E1_paywall_price,conversion to paid (per paywall view),-13% relative; x1.30 extra for treatment in th...
1,E1_paywall_price,price,+33% on every plan ($14.99 -> $19.99 weekly)
2,E1_paywall_price,first renewal,logit -0.15 (slightly fewer renewals); refunds...
3,E2_onboarding,drop-off per onboarding step,-10% relative (small real gain); Android treat...
4,E3_smart_picks,"matches, whole-market effect",+4% when every user is treated (tau = 0.04); t...
5,E4a-e,return within 7 days,no effect in any of the five tests (assignment...
6,E5_profile_prompt,activation,no effect (assignment seed 7 chosen so an earl...
7,E6_icebreakers,"messages sent, 1-14 Oct",+5% for treated users
8,opportunity,Android photo-upload step,pass rate 0.66 vs 0.89 on iOS
9,opportunity,no like received on day 0,day-1 return probability x0.52
